# Ejemplo 1 - Agente de IA básico

## Propósito didáctico
Este notebook está pensado para **explicar paso a paso** cómo construir un agente conversacional sencillo con **LangChain + Groq** usando el enfoque moderno basado en mensajes.

## Competencias que se trabajan
- Comprender qué hace un modelo de chat.
- Distinguir entre **prompt del sistema**, **mensaje del usuario** e **historial**.
- Implementar una memoria conversacional simple sin usar `ConversationChain`.
- Probar una interacción en consola y reflexionar sobre sus límites.

## Antes de comenzar: ¿qué problema resolvemos?
Un agente conversacional básico recibe texto del usuario, lo envía al modelo y devuelve una respuesta.  
Para que no “olvide” lo dicho en el turno anterior, guardamos el historial de mensajes.

### Flujo conceptual
1. El usuario escribe una pregunta.
2. El sistema añade esa pregunta al historial.
3. El modelo genera una respuesta.
4. La respuesta también se guarda.
5. El ciclo se repite.

### Diferencia con el enfoque antiguo
Antes era común usar `ConversationChain` y `ConversationBufferMemory`.  
En el enfoque actual, para casos sencillos, resulta más claro trabajar directamente con una **lista de mensajes**.

## 1. Instalación
Ejecuta esta celda una sola vez en Colab o Jupyter.

In [ ]:
#
!pip install -qU langchain langchain-core langchain-groq

## 2. Configurar credenciales
La clave API se solicita con `getpass` para evitar escribirla directamente en el notebook.

**Buena práctica:** no compartas notebooks con llaves incrustadas en el código.

In [ ]:
#
import os
import getpass

# Dos opciones para cargar el API Key
#os.environ["GROQ_API_KEY"] = "..."

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("🔑 Ingresa tu Groq API key: ")


## 3. Inicializar el modelo
Aquí definimos:
- el **modelo** que se usará;
- la **temperatura**, que controla qué tan creativas o estables serán las respuestas;
- el **mensaje del sistema**, que orienta el comportamiento general del asistente.

### Pregunta de reflexión
¿Qué cambiaría si el mensaje del sistema dijera: *“responde como tutor de estadística”*?

In [ ]:
#
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

MODEL_NAME = "openai/gpt-oss-120b"
SYSTEM_PROMPT = """Eres un asistente útil, claro y preciso.
Responde en español salvo que el usuario pida otro idioma.
Si no sabes algo, dilo con honestidad."""

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    max_tokens=1024,
    max_retries=2,
)


## 4. Construir la memoria conversacional
La “memoria” no es magia: es simplemente una lista de mensajes que reenviamos al modelo en cada turno.

### ¿Por qué limitar el historial?
Si la conversación crece demasiado:
- aumenta el costo computacional;
- puede volverse más lenta;
- el modelo podría distraerse con información poco relevante.

Por eso conservamos solo los últimos turnos.

In [ ]:
#
MAX_TURNS = 8  # conserva los últimos 8 turnos usuario-asistente

messages = [SystemMessage(content=SYSTEM_PROMPT)]

#
def _trim_history(history, max_turns=MAX_TURNS):
    system = history[:1]
    convo = history[1:]
    max_messages = max_turns * 2
    if len(convo) > max_messages:
        convo = convo[-max_messages:]
    return system + convo


## 5. Definir la función de conversación
Esta función encapsula el comportamiento principal:
- agrega la entrada del usuario;
- consulta al modelo;
- guarda la respuesta;
- devuelve texto listo para mostrar.

### Ventaja pedagógica
Este diseño permite ver con claridad **dónde entra** el usuario, **dónde responde** el modelo y **cómo se conserva** el contexto.

In [ ]:
def conversation(user_input: str) -> str:
    global messages

    user_input = user_input.strip()
    if not user_input:
        return "Por favor escribe una pregunta o instrucción."

    messages.append(HumanMessage(content=user_input))
    messages = _trim_history(messages)

    try:
        response = llm.invoke(messages)
        messages.append(AIMessage(content=response.content))
        messages = _trim_history(messages)
        return response.content
    except Exception as e:
        # Removemos el último mensaje del usuario si falló la llamada
        if messages and isinstance(messages[-1], HumanMessage):
            messages.pop()
        return f"Ocurrió un error al consultar el modelo: {e}"


## 6. Ejecutar el agente en modo interactivo
Este bloque abre una conversación en consola.  
Incluye algunos comandos útiles, por ejemplo:
- `salir` o `exit` para terminar;
- `reset` para borrar el historial;
- `historial` para inspeccionar la conversación almacenada.

### Sugerencia para clase
Prueba tres tipos de preguntas:
1. una factual;
2. una creativa;
3. una que dependa del contexto anterior.

In [ ]:
#
print("=== Agente de IA Básico ===")
print("Escribe 'salir' para terminar, 'reset' para reiniciar o 'historial' para inspeccionar la memoria.")

while True:
    pregunta = input("Tú: ").strip()

    if pregunta.lower() in ["salir", "exit"]:
        print("Agente: ¡Hasta luego!")
        break

    if pregunta.lower() == "reset":
        messages = [SystemMessage(content=SYSTEM_PROMPT)]
        print("Agente: He reiniciado el historial de la conversación.")
        continue

    if pregunta.lower() == "historial":
        print("=== Historial actual ===")
        for msg in messages:
            role = type(msg).__name__.replace("Message", "")
            print(f"[{role}] {msg.content}")
        continue

    respuesta = conversation(pregunta)
    print(f"Agente: {respuesta}")


## 7. Ver la cadena de pensamiento

`openai/gpt-oss-120b` es un **modelo de razonamiento**: antes de contestar genera un borrador
donde delibera. Ese borrador viaja **aparte** de la respuesta, y LangChain lo guarda en los
metadatos del mensaje.

El problema práctico es que cada proveedor lo pone en una llave distinta y los nombres cambian
entre versiones de la librería. Por eso la función de abajo busca en los lugares conocidos y
devuelve `None` si no encuentra nada, en vez de tronar.

In [ ]:
import textwrap


def extraer_razonamiento(mensaje):
    """Devuelve la cadena de pensamiento de un mensaje de LangChain, o None si no la hay.

    Buscamos en varios lugares a propósito: el nombre del campo cambia entre
    proveedores y entre versiones de langchain-groq.
    """
    extras = {}
    extras.update(getattr(mensaje, "additional_kwargs", None) or {})
    extras.update(getattr(mensaje, "response_metadata", None) or {})

    for clave in ("reasoning_content", "reasoning", "thinking"):
        valor = extras.get(clave)
        if isinstance(valor, str) and valor.strip():
            return valor.strip()

    # Algunos modelos lo devuelven como bloques dentro del propio contenido.
    contenido = getattr(mensaje, "content", None)
    if isinstance(contenido, list):
        piezas = [b.get("reasoning") or b.get("text", "")
                  for b in contenido
                  if isinstance(b, dict) and b.get("type") in ("reasoning", "thinking")]
        piezas = [p for p in piezas if p]
        if piezas:
            return "\n".join(piezas)
    return None


def preguntar_mostrando_pensamiento(pregunta: str):
    """Hace una pregunta suelta (sin tocar la memoria) y muestra las dos capas."""
    respuesta = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=pregunta),
    ])

    pensamiento = extraer_razonamiento(respuesta)

    print("PENSAMIENTO  (el borrador interno; el usuario no lo ve)")
    print("=" * 78)
    print(textwrap.fill(pensamiento, 78) if pensamiento
          else "(este modelo no expuso cadena de pensamiento)")

    print("\nRESPUESTA  (esto es lo que recibe el usuario)")
    print("=" * 78)
    print(textwrap.fill(str(respuesta.content), 78))
    return respuesta


_ = preguntar_mostrando_pensamiento(
    "Un grupo de 30 alumnos entrega 2 tareas cada uno. Si reviso 12 tareas por hora "
    "y ya llevo 18 revisadas, ¿cuántas horas me faltan?"
)

Elegimos a propósito una pregunta con varios pasos: en el borrador se ve al modelo hacer las
cuentas —60 tareas en total, menos 18, entre 12— antes de escribir el resultado. Con una pregunta
trivial el borrador es de una línea, o ni siquiera aparece.

> **Tres advertencias.**
>
> 1. **No es una auditoría.** El borrador lo genera el mismo mecanismo que la respuesta: es
>    *plausible*, no necesariamente *fiel* a lo que ocurrió dentro del modelo. Sirve para depurar
>    y para entender, no como prueba de nada.
> 2. **No se lo muestres al usuario final.** Contiene hipótesis descartadas y cifras tentativas
>    que, fuera de contexto, se leen como afirmaciones del sistema.
> 3. **Se paga.** Esos tokens se cobran como tokens de salida, que son los más caros.
>
> Fíjate también en que **el borrador no entra a la memoria**: en `conversation()` guardamos solo
> `response.content`. Es lo correcto — reenviar los borradores en cada turno multiplicaría el
> costo sin mejorar las respuestas.

## 8. ¿Qué aprendimos?
Con este ejemplo ya puedes identificar los componentes mínimos de un agente conversacional:
- un modelo;
- un prompt base;
- un historial;
- una función de interacción.

## 9. Limitaciones del agente básico
Este agente todavía:
- no usa herramientas externas;
- no consulta internet;
- no ejecuta cálculos complejos por sí solo;
- puede equivocarse o inventar información.

## 10. Actividad sugerida
Modifica el `SYSTEM_PROMPT` para convertir el agente en uno de estos perfiles:
- tutor de programación;
- asistente académico;
- orientador de escritura.

Después, compara cómo cambia el estilo de sus respuestas.

## 11. Ideas para extenderlo
- Añadir herramientas.
- Guardar historial en archivo o base de datos.
- Agregar streaming de tokens.
- Integrar RAG con documentos propios.